# Aula 4 — População 2010 × 2022 (Censo IBGE)

Tabela do IBGE com a população de cada município em **2010** e no **Censo 2022**.

**Etapas:**
1. Ler a tabela
2. Criar uma tabela de população **agregada por estado** e ordenar pelo que mais cresceu entre 2010 e 2022 → salvar em CSV
3. Fazer o mesmo **por município** → salvar em CSV

In [1]:
# No terminal, uma vez: python -m pip install pandas openpyxl
import pandas as pd

In [2]:
# arquivo na mesma pasta do notebook
path = "CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"

## 4 - Lendo a tabela

Olhando o arquivo bruto: as primeiras linhas têm título e cabeçalho, e o final tem notas e fonte.

In [3]:
bruto = pd.read_excel(path, header=None)
bruto

,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Censo Demográfico 2022: População e Domicílios...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
3,NaN,RO,11,00015,Alta Floresta D'Oeste,24392,24392,21494
4,NaN,RO,11,00023,Ariquemes,90353,90353,96833
...,...,...,...,...,...,...,...,...
5572,NaN,DF,53,00108,Brasília,2570160,2572159,2817381
5573,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5574,NaN,Nota: Para o cálculo das taxas de crescimento ...,NaN,NaN,NaN,NaN,NaN,NaN
5575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Limpando

- Coluna 0: vazia → descartada
- Linhas 0–2: título e cabeçalho | 3–5572: os 5.570 municípios | 5573+: nota e fonte

A tabela tem **duas colunas de 2010**:
- `pop_2010_sinopse`: população recenseada em 2010
- `pop_2010`: população de 2010 **compatibilizada** com os limites territoriais de 2022

Segundo a nota do próprio IBGE, a comparação com 2022 deve usar a população **compatibilizada**, para que mudanças de limite entre municípios não pareçam crescimento ou perda de população. Por isso, as contas usam `pop_2010`.

In [4]:
populacao = bruto.iloc[3:5573, 1:].copy()
populacao.columns = [
    "uf",
    "cod_uf",
    "cod_munic",
    "municipio",
    "pop_2010_sinopse",
    "pop_2010",
    "pop_2022",
]
populacao = populacao.reset_index(drop=True)

for col in ["pop_2010_sinopse", "pop_2010", "pop_2022"]:
    populacao[col] = pd.to_numeric(populacao[col], errors="coerce").astype(int)

print(f"Municípios: {len(populacao)} | Estados: {populacao['uf'].nunique()}")
populacao.head(10)

Municípios: 5570 | Estados: 27


,uf,cod_uf,cod_munic,municipio,pop_2010_sinopse,pop_2010,pop_2022
0,RO,11,00015,Alta Floresta D'Oeste,24392,24392,21494
1,RO,11,00023,Ariquemes,90353,90353,96833
2,RO,11,00031,Cabixi,6313,6313,5351
3,RO,11,00049,Cacoal,78574,78574,86887
4,RO,11,00056,Cerejeiras,17029,17029,15890
5,RO,11,00064,Colorado do Oeste,18591,18591,15663
6,RO,11,00072,Corumbiara,8783,8783,7519
7,RO,11,00080,Costa Marques,13678,13678,12627
8,RO,11,00098,Espigão D'Oeste,28729,28729,29414
9,RO,11,00106,Guajará-Mirim,41656,41656,39387


## 5 - Tabela de população agregada por estado

In [5]:
pop_estado = (
    populacao.groupby("uf")[["pop_2010", "pop_2022"]]
    .sum()
    .reset_index()
)
pop_estado

,uf,pop_2010,pop_2022
0,AC,733559,830018
1,AL,3120887,3127683
2,AM,3483985,3941613
3,AP,669526,733759
4,BA,14017071,14141626
5,CE,8451644,8794957
6,DF,2572159,2817381
7,ES,3514952,3833712
8,GO,6001789,7056495
9,MA,6574789,6776699


## 6 - Ordenando pelos estados que mais cresceram entre 2010 e 2022

- `diferenca`: crescimento em número de pessoas (`pop_2022 - pop_2010`)
- `crescimento_%`: crescimento percentual, para comparar estados de tamanhos diferentes

A ordenação é pela `diferenca` (quem ganhou mais habitantes).

In [6]:
pop_estado["diferenca"] = pop_estado["pop_2022"] - pop_estado["pop_2010"]
pop_estado["crescimento_%"] = (pop_estado["diferenca"] / pop_estado["pop_2010"] * 100).round(2)

pop_estado = pop_estado.sort_values("diferenca", ascending=False).reset_index(drop=True)
pop_estado

,uf,pop_2010,pop_2022,diferenca,crescimento_%
0,SP,41262199,44411238,3149039,7.63
1,SC,6248436,7610361,1361925,21.80
2,GO,6001789,7056495,1054706,17.57
3,PR,10444526,11444380,999854,9.57
4,MG,19597330,20539989,942659,4.81
5,MT,3035122,3658649,623527,20.54
6,PA,7581051,8120131,539080,7.11
7,AM,3483985,3941613,457628,13.14
8,CE,8451644,8794957,343313,4.06
9,ES,3514952,3833712,318760,9.07


## 7 - Salvando a tabela de estados em CSV

In [7]:
pop_estado.to_csv("populacao_por_estado.csv", sep=";", index=False, encoding="utf-8-sig")
print("Arquivo salvo: populacao_por_estado.csv")

Arquivo salvo: populacao_por_estado.csv


## 8 - Agora por município

Cada linha da tabela original já é um município, então não precisa de `groupby`.

⚠️ Agrupar só pelo **nome** seria um erro, porque existem municípios com o mesmo nome em estados diferentes (ex.: *Bom Jesus*, *Santa Helena*). Somar pelo nome misturaria cidades diferentes. Por isso a tabela mantém a coluna `uf` junto com o nome.

In [8]:
repetidos = populacao["municipio"].value_counts()
print(f"Nomes de município que aparecem em mais de um estado: {(repetidos > 1).sum()}")
repetidos.head()

Nomes de município que aparecem em mais de um estado: 232


municipio
Bom Jesus       5
São Domingos    5
Bonito          4
Santa Helena    4
Santa Inês      4
Name: count, dtype: int64

In [9]:
pop_municipio = populacao[["uf", "municipio", "pop_2010", "pop_2022"]].copy()
pop_municipio

,uf,municipio,pop_2010,pop_2022
0,RO,Alta Floresta D'Oeste,24392,21494
1,RO,Ariquemes,90353,96833
2,RO,Cabixi,6313,5351
3,RO,Cacoal,78574,86887
4,RO,Cerejeiras,17029,15890
...,...,...,...,...
5565,GO,Vianópolis,12548,14956
5566,GO,Vicentinópolis,7373,8768
5567,GO,Vila Boa,4735,4215
5568,GO,Vila Propício,5145,5815


## 9 - Ordenando pelos municípios que mais cresceram entre 2010 e 2022

In [10]:
pop_municipio["diferenca"] = pop_municipio["pop_2022"] - pop_municipio["pop_2010"]
pop_municipio["crescimento_%"] = (pop_municipio["diferenca"] / pop_municipio["pop_2010"] * 100).round(2)

pop_municipio = pop_municipio.sort_values("diferenca", ascending=False).reset_index(drop=True)

print("10 municípios que mais cresceram:")
pop_municipio.head(10)

10 municípios que mais cresceram:


,uf,municipio,pop_2010,pop_2022,diferenca,crescimento_%
0,AM,Manaus,1802014,2063689,261675,14.52
1,DF,Brasília,2572159,2817381,245222,9.53
2,SP,São Paulo,11253503,11451999,198496,1.76
3,SP,Sorocaba,586816,723682,136866,23.32
4,GO,Goiânia,1301912,1437366,135454,10.40
5,RR,Boa Vista,284313,413486,129173,45.43
6,SC,Florianópolis,421240,537211,115971,27.53
7,PA,Parauapebas,153908,267836,113928,74.02
8,MS,Campo Grande,786774,898100,111326,14.15
9,PB,João Pessoa,723515,833932,110417,15.26


In [11]:
print("10 municípios que mais perderam população:")
pop_municipio.tail(10)

10 municípios que mais perderam população:


,uf,municipio,pop_2010,pop_2022,diferenca,crescimento_%
5560,CE,Fortaleza,2459712,2428708,-31004,-1.26
5561,RJ,Duque de Caxias,855088,808161,-46927,-5.49
5562,PE,Recife,1537704,1488920,-48784,-3.17
5563,RN,Natal,803739,751300,-52439,-6.52
5564,MG,Belo Horizonte,2375609,2315560,-60049,-2.53
5565,RS,Porto Alegre,1409351,1332845,-76506,-5.43
5566,PA,Belém,1393399,1303403,-89996,-6.46
5567,RJ,São Gonçalo,999728,896744,-102984,-10.30
5568,RJ,Rio de Janeiro,6320446,6211223,-109223,-1.73
5569,BA,Salvador,2675656,2417678,-257978,-9.64


## 10 - Salvando a tabela de municípios em CSV

In [12]:
pop_municipio.to_csv("populacao_por_municipio.csv", sep=";", index=False, encoding="utf-8-sig")
print("Arquivo salvo: populacao_por_municipio.csv")

Arquivo salvo: populacao_por_municipio.csv


## Resumo

In [13]:
brasil_2010 = pop_estado["pop_2010"].sum()
brasil_2022 = pop_estado["pop_2022"].sum()

print(f"Brasil 2010: {brasil_2010:,}".replace(",", "."))
print(f"Brasil 2022: {brasil_2022:,}".replace(",", "."))
print(f"Crescimento: {brasil_2022 - brasil_2010:,} pessoas ({(brasil_2022 / brasil_2010 - 1) * 100:.2f}%)".replace(",", "."))
print()
top_uf = pop_estado.iloc[0]
top_mun = pop_municipio.iloc[0]
print(f"Estado que mais cresceu: {top_uf['uf']} (+{top_uf['diferenca']:,} pessoas)".replace(",", "."))
print(f"Município que mais cresceu: {top_mun['municipio']} - {top_mun['uf']} (+{top_mun['diferenca']:,} pessoas)".replace(",", "."))
encolheram = (pop_estado["diferenca"] < 0).sum()
print(f"Estados que perderam população: {encolheram}")
print(f"Municípios que perderam população: {(pop_municipio['diferenca'] < 0).sum()}")

Brasil 2010: 190.755.799
Brasil 2022: 203.080.756
Crescimento: 12.324.957 pessoas (6.46%)

Estado que mais cresceu: SP (+3.149.039 pessoas)
Município que mais cresceu: Manaus - AM (+261.675 pessoas)
Estados que perderam população: 0
Municípios que perderam população: 2395
